# Chrome Dino RL Agent

## 프로젝트 개요
본 프로젝트는 구글 크롬 브라우저의 오프라인 공룡 게임(Chrome Dinosaur Game)을 플레이하는 심층 강화학습(Deep Reinforcement Learning) 에이전트입니다. 화면의 픽셀 데이터를 직접 시각적 상태(State)로 입력받아 최적의 행동(Action: 점프, 숙이기, 대기)을 도출하도록 설계되었습니다. 게임 진행에 따른 환경의 비정상성(Non-stationary, 속도 증가)에 대응하기 위해 프레임 차분(Difference) 기법과 D3QN(Dueling Double DQN) 아키텍처, 그리고 TD-Error 기반의 하이브리드 버퍼를 채택했습니다.

## 주요 기능 및 특징
- **실시간 화면 인식 및 차분 이미지 전처리**: `mss`와 OpenCV 프레임워크를 결합해 복수 모니터 환경에서도 템플릿 매칭 기반으로 단일 게임 영역을 자동 탐색합니다. 배경 노이즈를 제거하고 객체의 가속도를 네트워크에 전달하기 위해 연속된 프레임 간의 차분(Difference) 이미지를 추출하여 4-Frame Stacking을 수행합니다.
- **Dueling Double DQN 네트워크 구조**: 상태의 가치(Value)와 행동의 이점(Advantage)을 분리하여 추산하는 Dueling 구조와 타겟 네트워크의 과대평가를 방지하는 Double DQN을 결합하여 모델의 학습 수렴 안정성을 확보했습니다.
- **하이브리드 듀얼 리플레이 버퍼 (Hybrid Dual Replay Buffer)**: 데이터 불균형 문제를 해결하기 위해 이벤트(사망, 행동) 데이터를 강제로 오버샘플링함과 동시에, 일반 주행 데이터 중 TD-Error가 높은 경험을 우선도 버퍼(Priority Buffer)로 반송(Recycling)하는 독자적인 메모리 구조를 구현했습니다. 높은 연산 부하를 동반하는 SumTree 없이 PER(Prioritized Experience Replay)의 효과를 달성합니다.
- **정규화된 보상 체계 및 로깅**: Q-Value 발산(Explosion)을 방지하기 위해 보상 스케일을 [-1.0, 1.0] 범위로 정규화(Reward Scaling)했습니다. TensorBoard 로깅 시스템 모듈화를 통하여 생존 시간, 리워드 누적 합, 탐험률(Epsilon) 감소 지표 및 네트워크의 최고 Q값을 실시간으로 추적 및 시각화합니다.

## 디렉토리 구조 및 핵심 모듈
- `train_agent.py` : 강화학습 프로세스 메인 엔트리 스크립트. Dataclass(`AgentConfig`) 기반 하이퍼파라미터 주입, 에피소드 초기화 및 진행 루프 관리와 최적 가중치 갱신 작업을 담당.
- `test_agent.py` : 훈련 완료된 `.pth` 모델을 호출해 의사결정 정책 평가용 무탐험(Epsilon = 0) 플레이 스크립트.
- `environments/environment.py` : 행동 수행, 보상 체계(Reward Function) 산정, Terminal State 판별 등 에이전트와 환경 상호작용 인터페이스 구현 래퍼.
- `environments/vision.py` : 화면 픽셀 스크랩 및 OpenCV 연산. 논리/물리 해상도 매핑 연산, 프레임 차분(Difference) 추출 및 텐서 데이터 스태킹 수행.
- `environments/d3qn.py` / `dqn.py` : PyTorch를 통한 신경망 모델 아키텍쳐. Feature Extraction을 위한 합성곱 계층과 가치/행동 분리형 전결합(FC) 계층 지원.
- `environments/actions.py` : `pyautogui` I/O 바인딩. 동시 입력 및 키업/다운 이벤트를 통해 상태머신 기반 조작 관리.
- `trainer/train_buffer.py` / `replay_buffer.py` : Transition 기록을 수집하고, 손실 역전파(Backpropagation) 최적화 및 TD-Recycling을 수행하는 오프폴리시(Off-policy) 학습 지원 모듈.

## 환경 요구사항
Python 환경 하에 구성 가능하며, 다음과 같은 주요 패키지가 상호 호환되어야 합니다.

```bash
matplotlib
mss
numpy
pyautogui
torch
opencv-python
tensorboard
```

---

## 📂 프로젝트 구조

```text
dino_rl_agent/
├── environments/          # 환경 상호작용 및 비전/모델 아키텍처
│   ├── environment.py     # 에이전트-게임 간 Step, Reward 평가 래퍼
│   ├── vision.py          # 화면 캡처, 차분 이미지 전처리 및 스태킹
│   ├── actions.py         # pyautogui 기반 키보드 제어 로직
│   ├── dqn.py / d3qn.py   # CNN 기반 DQN 및 Dueling DQN 모델 정의
│   └── template.png       # 모니터 자동 인식을 위한 템플릿 매칭 원본
├── trainer/               # 강화학습 훈련 및 메모리 코어
│   ├── train_buffer.py    # 미니배치 샘플링, Loss 계산, 역전파 및 TD-Recycling
│   └── replay_buffer.py   # 우선도/노멀 큐 분리를 통한 듀얼 메모리 버퍼
├── utils/                 # 로깅 및 도구
│   ├── visualize.py       # Matplotlib 기반 학습 결과 차트 생성
│   ├── record_play.py     # 훈련된 모델의 플레이 화면 영상 녹화
│   └── find_monitor.py    # 게임오버 픽셀 좌표 디버깅 툴
├── train_agent.py         # 🚀 메인 훈련 스크립트 (Dataclass 설정 및 루프)
├── test_agent.py          # 🎯 학습 완료 모델 실전 성능 평가 (Epsilon=0)
└── config.py              # (Deprecated) 기존 환경 설정 파일